# How to use the API with the Rasterio Backend

Since most of the API documentation uses the `xarray` backend, this notebook provides an example of how to use most endpoints with the `rasterio` backend.

See also the [API docs](https://staging.openveda.cloud/api/titiler-cmr/api.html).

## Setup

In [ ]:
import json

import earthaccess
import httpx

from folium import Map, TileLayer

# titiler_endpoint = (
#     "https://staging.openveda.cloud/api/titiler-cmr"  # VEDA staging endpoint
# )
titiler_endpoint = (
    "https://v4jec6i5c0.execute-api.us-west-2.amazonaws.com"  # dev endpoint
)

## Identify the dataset
The Harmonized Landsat Sentinel-2 dataset is available in two collections in CMR. This example will use data from the `HLSL30.002` (Landsat) dataset.
You can find the `HLSL30.002` dataset using the earthaccess.search_datasets function.

In [ ]:
datasets = earthaccess.search_datasets(keyword="HLSL30")
ds = datasets[0]

collection_concept_id = ds["meta"]["concept-id"]
print("Collection Concept-Id: ", collection_concept_id)
print("Abstract: ", ds["umm"]["Abstract"])

## Explore the collection using the `/compatibility` endpoint

In [ ]:
compatibility_response = httpx.get(
    f"{titiler_endpoint}/compatibility",
    params={"collection_concept_id": collection_concept_id},
    timeout=None,
).json()

print(json.dumps(compatibility_response, indent=2))

The details from the sample granule show that it has 18 assets (you would need to look more into what each of the assets represents). To properly configure the assets for titiler-cmr we will need to use the `bands_regex` parameter to identify the bands that we want to be available for visualizations. The `datetime` key shows the reported temporal range from CMR which indicates that the dataset has granules from `2015-11-28` to present.

## Display tiles in an interactive map

The `/tilejson.json` endpoint will provide a parameterized `xyz` tile URL that can be added to an interactive map.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/rasterio/WebMercatorQuad/tilejson.json",
    params=(
        ("collection_concept_id", collection_concept_id),
        # Temporal range in form of `start_date/end_date`
        ("temporal", "2024-10-01T00:00:00Z/2024-10-31T23:59:59Z"),
        # Sort by cloud cover in ascending order to render less cloudy images!
        ("sort_key", "cloud_cover"),
        # We know that the HLS collection dataset is stored as File per Band
        # so we need to pass a `asset_regex` option to assign `assets` to each URL
        ("assets_regex", "B[0-9][0-9]"),
        # True Color Image B04,B03,B02
        ("assets", "B04"),
        ("assets", "B03"),
        ("assets", "B02"),
        # The data is in type of Uint16 so we need to apply some
        # rescaling/color_formula in order to create PNGs
        ("color_formula", "Gamma RGB 3.5 Saturation 1.7 Sigmoidal RGB 15 0.35"),
        # We need to set min/max zoom because we don't want to use lowerzoom level (e.g 0)
        # which will results in useless large scale query
        ("minzoom", 8),
        ("maxzoom", 13),
    ),
    timeout=None,
).json()

print(r)

In [ ]:
bounds = r["bounds"]
m = Map(location=(47.590266824611675, -91.03729840730689), zoom_start=r["maxzoom"] - 2)

TileLayer(
    tiles=r["tiles"][0],
    opacity=1,
    attr="NASA",
).add_to(m)
m

### Render NDVI using the `expression` parameter
The `expression` parameter can be used to render images from an expression of a combination of the individual `bands`.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/rasterio/WebMercatorQuad/tilejson.json",
    params=(
        ("collection_concept_id", collection_concept_id),
        # Temporal range in form of `start_date/end_date`
        ("temporal", "2024-06-01T00:00:00Z/2024-06-30T23:59:59Z"),
        ("sort_key", "cloud_cover"),
        # We know that the HLS collection dataset is stored as File per Band
        # so we need to pass a `assets_regex` option to assign `assets` to each URL
        ("assets_regex", "B[0-9][0-9]"),
        ("asset_as_band", True),
        # NDVI
        ("expression", "(B05-B04)/(B05+B04)"),
        # Need red (B04) and nir (B05) for NDVI
        ("assets", "B05"),
        ("assets", "B04"),
        # The data is in type of Uint16 so we need to apply some
        # rescaling/color_formula in order to create PNGs
        ("colormap_name", "greens"),
        ("rescale", "-1,1"),
        # We need to set min/max zoom because we don't want to use lowerzoom level (e.g 0)
        # which will results in useless large scale query
        ("minzoom", 8),
        ("maxzoom", 13),
    ),
    timeout=None,
).json()

m = Map(location=(47.9221313337365, -91.65432884883238), zoom_start=r["maxzoom"] - 1)


TileLayer(
    tiles=r["tiles"][0],
    opacity=1,
    attr="NASA",
).add_to(m)

m

## Visualize a specific granule

TiTiler-CMR allows you to visualize a specific granule with the `granule_ur` query parameter.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/rasterio/WebMercatorQuad/tilejson.json",
    params=(
        ("collection_concept_id", "C2021957657-LPCLOUD"),
        # Unique granule id (granule_ur)
        ("granule_ur", "HLS.L30.T44RQR.2026061T045429.v2.0"),
        # We know that the HLS collection dataset is stored as File per Band
        # so we need to pass a `asset_regex` option to assign `assets` to each URL
        ("assets_regex", "B[0-9][0-9]"),
        # True Color Image B04,B03,B02
        ("assets", "B04"),
        ("assets", "B03"),
        ("assets", "B02"),
        # The data is in type of Uint16 so we need to apply some
        # rescaling/color_formula in order to create PNGs
        ("color_formula", "Gamma RGB 3.5 Saturation 1.7 Sigmoidal RGB 15 0.35"),
    ),
    timeout=None,
).json()

print(r)

m = Map(location=(r["center"][1], r["center"][0]), zoom_start=9)

TileLayer(
    tiles=r["tiles"][0],
    opacity=1,
    attr="NASA",
).add_to(m)

m

## GeoJSON Statistics
The `/statistics` endpoint can be used to get summary statistics for a geojson `Feature` or `FeatureCollection`.

In [ ]:
geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "coordinates": [
                    [
                        [-91.65432884883238, 47.9221313337365],
                        [-91.65432884883238, 47.86503396133904],
                        [-91.53842043960762, 47.86503396133904],
                        [-91.53842043960762, 47.9221313337365],
                        [-91.65432884883238, 47.9221313337365],
                    ]
                ],
                "type": "Polygon",
            },
        }
    ],
}

In [ ]:
r = httpx.post(
    f"{titiler_endpoint}/rasterio/statistics",
    params=(
        ("collection_concept_id", collection_concept_id),
        # Temporal range in form of `start_date/end_date`
        ("temporal", "2024-07-01T00:00:00Z/2024-07-10T23:59:59Z"),
        # We know that the HLS collection dataset is stored as File per Band
        # so we need to pass a `band_regex` option to assign `bands` to each URL
        ("assets_regex", "B[0-9][0-9]"),
        # NDVI
        ("expression", "(b1-b2)/(b1+b2)"),
        # Need red (B04) and nir (B05) for NDVI
        ("assets", "B05"),
        ("assets", "B04"),
    ),
    json=geojson,
    timeout=None,
).json()

print(json.dumps(r, indent=2))